# DFU V3.1 Missing-38 Durable Hotfix
Fixes the imported metadata-only checkpoint crash. Existing V2/V3 evidence is preserved.


In [ ]:
import ast, base64, gzip, hashlib, urllib.request

PAYLOAD_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/ea0b237b8276792d7120b11c6a28387c480bb846/notebooks/v3_loader_payload/DFU_V3_Missing38_Durable_OneCell.py.gz.b64"
EXPECTED_ORIGINAL_SHA256 = "32af222793fd8a8d559969e48f8c886e0a43e28e184be90021475c8723a92c0b"
EXPECTED_PATCHED_SHA256 = "532dcea7cb274459e568a9935ca19dbd6f305d7152e2cc02c02bfd7bd95c9d51"
OLD_BLOCK = '    def wrapped_train_trial(*args: Any, **kwargs: Any):\n        metrics, predictions = original_train_trial(*args, **kwargs)\n        trial = Path(kwargs["trial"])\n        run = Path(kwargs["run"])\n        checked = validate_trial_artifacts(\n            trial,\n            expected_model=kwargs["model_key"],\n            expected_seed=kwargs["seed"],\n            expected_fold_zero_based=kwargs["fold"],\n        )\n        _append_evidence_ledger(run, checked)\n        durable_metrics, durable_predictions = rebuild_durable_aggregates(run)\n        _atomic_csv(run / "tables" / "fold_seed_metrics.csv", durable_metrics)\n        _atomic_csv(run / "tables" / "all_oof_predictions.csv", durable_predictions)\n        _atomic_json(\n            run / "DURABLE_PROGRESS.json",\n            {\n                "completed_trials": int(len(durable_metrics)),\n                "expected_trials": 45,\n                "prediction_rows": int(len(durable_predictions)),\n                "last_verified": {\n                    "model_key": checked["model_key"],\n                    "seed": checked["seed"],\n                    "outer_fold": checked["outer_fold"],\n                },\n                "updated_at_ns": time.time_ns(),\n            },\n        )\n        return metrics, predictions\n'
NEW_BLOCK = '    def wrapped_train_trial(*args: Any, **kwargs: Any):\n        trial = Path(kwargs["trial"])\n        run = Path(kwargs["run"])\n\n        # V3.1 hotfix: recovered/completed evidence may intentionally have a\n        # metadata-only portable checkpoint. Never send those trials into the\n        # V2 retention path, which requires a full state_dict and caused the\n        # pre-training crash seen in V3.\n        try:\n            checked = validate_trial_artifacts(\n                trial,\n                expected_model=kwargs["model_key"],\n                expected_seed=kwargs["seed"],\n                expected_fold_zero_based=kwargs["fold"],\n            )\n        except Exception:\n            checked = None\n\n        if checked is not None:\n            metrics = checked["metrics"]\n            predictions = checked["predictions"]\n            print(\n                "RESUME: durable completed evidence skipped — "\n                f"{kwargs[\'model_key\']} seed={kwargs[\'seed\']} fold={int(kwargs[\'fold\']) + 1}"\n            )\n        else:\n            metrics, predictions = original_train_trial(*args, **kwargs)\n            checked = validate_trial_artifacts(\n                trial,\n                expected_model=kwargs["model_key"],\n                expected_seed=kwargs["seed"],\n                expected_fold_zero_based=kwargs["fold"],\n            )\n\n        _append_evidence_ledger(run, checked)\n        durable_metrics, durable_predictions = rebuild_durable_aggregates(run)\n        _atomic_csv(run / "tables" / "fold_seed_metrics.csv", durable_metrics)\n        _atomic_csv(run / "tables" / "all_oof_predictions.csv", durable_predictions)\n        _atomic_json(\n            run / "DURABLE_PROGRESS.json",\n            {\n                "completed_trials": int(len(durable_metrics)),\n                "expected_trials": 45,\n                "prediction_rows": int(len(durable_predictions)),\n                "last_verified": {\n                    "model_key": checked["model_key"],\n                    "seed": checked["seed"],\n                    "outer_fold": checked["outer_fold"],\n                },\n                "updated_at_ns": time.time_ns(),\n                "hotfix": "v3.1-evidence-skip-before-v2-retention",\n            },\n        )\n        return metrics, predictions\n'

encoded = urllib.request.urlopen(PAYLOAD_URL, timeout=60).read().strip()
original_bytes = gzip.decompress(base64.b64decode(encoded, validate=True))
original_sha = hashlib.sha256(original_bytes).hexdigest()
if original_sha != EXPECTED_ORIGINAL_SHA256:
    raise RuntimeError(f"Original pinned payload SHA mismatch: {original_sha} != {EXPECTED_ORIGINAL_SHA256}")

script = original_bytes.decode("utf-8")
tree = ast.parse(script)
assignment = tree.body[0]
module_code = assignment.value.value
if module_code.count(OLD_BLOCK) != 1:
    raise RuntimeError(f"Hotfix target count must be 1, found {module_code.count(OLD_BLOCK)}")
module_code = module_code.replace(OLD_BLOCK, NEW_BLOCK, 1)
module_code = module_code.replace(
    'EXPERIMENT_ID = "dfu-v3-missing38-algorithm-349143-storage-v3"',
    'EXPERIMENT_ID = "dfu-v3.1-missing38-algorithm-349143-storage-hotfix"',
    1,
)
ast.parse(module_code)
lines = script.splitlines(keepends=True)
start = sum(len(x) for x in lines[:assignment.lineno - 1]) + assignment.col_offset
end = sum(len(x) for x in lines[:assignment.end_lineno - 1]) + assignment.end_col_offset
script = script[:start] + "MODULE_CODE = " + repr(module_code) + script[end:]
script = script.replace(
    "print('V3 durable patches: INSTALLED')",
    "print('V3.1 durable hotfix patches: INSTALLED')",
    1,
)
script = script.replace(
    "'base_commit': BASE_COMMIT,",
    "'base_commit': BASE_COMMIT,\n        'hotfix': 'v3.1-evidence-skip-before-v2-retention',",
    1,
)
patched_bytes = script.encode("utf-8")
patched_sha = hashlib.sha256(patched_bytes).hexdigest()
if patched_sha != EXPECTED_PATCHED_SHA256:
    raise RuntimeError(f"Patched payload SHA mismatch: {patched_sha} != {EXPECTED_PATCHED_SHA256}")
print("Pinned V3.1 hotfix payload verification: PASS")
exec(compile(script, "DFU_V3_1_Missing38_Durable_Hotfix.py", "exec"), globals())
